In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow import keras

2026-05-16 15:50:02.091427: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-16 15:50:02.134753: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-16 15:50:02.304278: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-16 15:50:02.305045: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-16 15:50:03.524878: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

In [2]:
df = pd.read_csv('sonar_dataset.csv', header=None)

print(df.head())
print(df.shape)

       0       1       2       3       4       5       6       7       8   \
0  0.0200  0.0371  0.0428  0.0207  0.0954  0.0986  0.1539  0.1601  0.3109   
1  0.0453  0.0523  0.0843  0.0689  0.1183  0.2583  0.2156  0.3481  0.3337   
2  0.0262  0.0582  0.1099  0.1083  0.0974  0.2280  0.2431  0.3771  0.5598   
3  0.0100  0.0171  0.0623  0.0205  0.0205  0.0368  0.1098  0.1276  0.0598   
4  0.0762  0.0666  0.0481  0.0394  0.0590  0.0649  0.1209  0.2467  0.3564   

       9   ...      51      52      53      54      55      56      57  \
0  0.2111  ...  0.0027  0.0065  0.0159  0.0072  0.0167  0.0180  0.0084   
1  0.2872  ...  0.0084  0.0089  0.0048  0.0094  0.0191  0.0140  0.0049   
2  0.6194  ...  0.0232  0.0166  0.0095  0.0180  0.0244  0.0316  0.0164   
3  0.1264  ...  0.0121  0.0036  0.0150  0.0085  0.0073  0.0050  0.0044   
4  0.4459  ...  0.0031  0.0054  0.0105  0.0110  0.0015  0.0072  0.0048   

       58      59  60  
0  0.0090  0.0032   R  
1  0.0052  0.0044   R  
2  0.0095  0.0078   

In [3]:
# Split input and output
x = df.drop(60, axis=1)
y = df[60]

In [4]:
# Convert labels (M and R) into 0 and 1
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [5]:
# Train and test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [7]:
# model without dropout layer
model = keras.Sequential([
    keras.layers.Dense(60, input_dim=60, activation='relu'),
    keras.layers.Dense(30, activation='relu'),
    keras.layers.Dense(15, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=100, batch_size=8)

Epoch 1/100
21/21 [==============================] - 1s 1ms/step - loss: 0.6999 - accuracy: 0.5301
Epoch 2/100
21/21 [==============================] - 0s 1ms/step - loss: 0.6218 - accuracy: 0.7349
Epoch 3/100
21/21 [==============================] - 0s 1ms/step - loss: 0.5525 - accuracy: 0.7952
Epoch 4/100
21/21 [==============================] - 0s 1ms/step - loss: 0.4812 - accuracy: 0.8373
Epoch 5/100
21/21 [==============================] - 0s 1ms/step - loss: 0.3998 - accuracy: 0.8735
Epoch 6/100
21/21 [==============================] - 0s 862us/step - loss: 0.3134 - accuracy: 0.9217
Epoch 7/100
21/21 [==============================] - 0s 884us/step - loss: 0.2316 - accuracy: 0.9458
Epoch 8/100
21/21 [==============================] - 0s 866us/step - loss: 0.1673 - accuracy: 0.9699
Epoch 9/100
21/21 [==============================] - 0s 986us/step - loss: 0.1223 - accuracy: 0.9759
Epoch 10/100
21/21 [==============================] - 0s 1ms/step - loss: 0.0913 - accuracy: 0.9759
E

In [8]:
model.evaluate(x_test, y_test)

2/2 [==============================] - 0s 6ms/step - loss: 0.3704 - accuracy: 0.9048


[0.3704172670841217, 0.9047619104385376]

In [9]:
y_pred = model.predict(x_test).reshape(-1)
print(y_pred[:10])

y_pred = np.round(y_pred)
print(y_pred[:10])

2/2 [==============================] - 0s 2ms/step
[9.6499988e-08 9.9988103e-01 9.9862576e-01 9.9999017e-01 1.0525398e-05
 9.7608191e-01 9.9667078e-01 2.3709445e-05 9.9999958e-01 8.2691223e-04]
[0. 1. 1. 1. 0. 1. 1. 0. 1. 0.]


In [10]:
from sklearn.metrics import confusion_matrix , classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.85      0.92        26
           1       0.80      1.00      0.89        16

    accuracy                           0.90        42
   macro avg       0.90      0.92      0.90        42
weighted avg       0.92      0.90      0.91        42



In [11]:
# model with dropout layer
model = keras.Sequential([
    keras.layers.Dense(256, activation='relu', input_shape=(x_train.shape[1],)),
    keras.layers.Dropout(0.2),

    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),

    keras.layers.Dense(64, activation='relu'),

    keras.layers.Dense(1, activation='sigmoid')
])

In [12]:
# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    x_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.1
)

Epoch 1/100
19/19 [==============================] - 1s 8ms/step - loss: 0.6505 - accuracy: 0.6107 - val_loss: 0.5812 - val_accuracy: 0.7647
Epoch 2/100
19/19 [==============================] - 0s 2ms/step - loss: 0.4009 - accuracy: 0.8792 - val_loss: 0.7057 - val_accuracy: 0.6471
Epoch 3/100
19/19 [==============================] - 0s 2ms/step - loss: 0.2739 - accuracy: 0.9128 - val_loss: 0.5442 - val_accuracy: 0.7647
Epoch 4/100
19/19 [==============================] - 0s 3ms/step - loss: 0.2050 - accuracy: 0.9128 - val_loss: 0.8720 - val_accuracy: 0.5882
Epoch 5/100
19/19 [==============================] - 0s 3ms/step - loss: 0.1635 - accuracy: 0.9329 - val_loss: 0.6760 - val_accuracy: 0.7059
Epoch 6/100
19/19 [==============================] - 0s 2ms/step - loss: 0.0631 - accuracy: 0.9799 - val_loss: 0.7298 - val_accuracy: 0.8235
Epoch 7/100
19/19 [==============================] - 0s 2ms/step - loss: 0.0450 - accuracy: 1.0000 - val_loss: 0.8943 - val_accuracy: 0.7059
Epoch 8/100
1

In [14]:
# Evaluate model
loss, accuracy = model.evaluate(x_test, y_test)

print("\nTest Accuracy:", accuracy)

# Predictions
y_pred = model.predict(x_test)
y_pred = (y_pred > 0.5).astype(int)

print(y_pred[:10])

2/2 [==============================] - 0s 4ms/step - loss: 0.7076 - accuracy: 0.9048

Test Accuracy: 0.9047619104385376
2/2 [==============================] - 0s 2ms/step
[[0]
 [1]
 [1]
 [1]
 [0]
 [1]
 [1]
 [0]
 [1]
 [0]]


In [15]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.88      0.92        26
           1       0.83      0.94      0.88        16

    accuracy                           0.90        42
   macro avg       0.90      0.91      0.90        42
weighted avg       0.91      0.90      0.91        42

